In [1]:
import torch
import numpy as np

In [2]:
from PIL import Image
from IPython.display import display
from __future__ import annotations

NUM_ROWS = 28
NUM_COLS = 28
NUM_LABELS = 10

class Data:
  def __init__(self, flattened_pixels: torch.Tensor, label: int):
    self.flattened_pixels = flattened_pixels
    self.label = label
  
  @classmethod
  def from_csv_row(cls, csv: list[int]) -> Data:
    label = csv[0]
    flattened_pixels = torch.tensor(csv[1:])
    flattened_pixels = flattened_pixels / 255
    return cls(flattened_pixels, label)
  
  def to_image(self) -> Image:
    numpy_array: np.ndarray = self.flattened_pixels.numpy()
    pixel_array = (numpy_array.reshape(NUM_ROWS, NUM_COLS) * 255).astype(np.uint8)
    return Image.fromarray(pixel_array, mode='L')
  
  def label_to_one_hot(self) -> torch.Tensor:
    result = torch.zeros(NUM_LABELS)
    result[self.label] = 1
    return result 

In [3]:
lines: list[str] = []
datas: list[Data] = []

with open('data/MNIST_CSV/mnist_train.csv') as f:
  lines = f.readlines()

for line in lines:
  datas.append(Data.from_csv_row([int(num) for num in line.split(',')]))

In [4]:
print(datas[0].label)
print(datas[0].label_to_one_hot())
display(datas[0].to_image())

5
tensor([0., 0., 0., 0., 0., 1., 0., 0., 0., 0.])


In [5]:
distinct_labels = set()
for data in datas:
  distinct_labels.add(data.label)
distinct_labels

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9}

In [6]:
from torch import nn

In [7]:
device = "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.neural_network = self.initialize_neural_network()

    def initialize_neural_network(self):
        init_layer = nn.Linear(28*28, 32)
        nn.init.xavier_uniform_(init_layer.weight)
        nn.init.zeros_(init_layer.bias)

        final_layer = nn.Linear(32, 10)
        nn.init.zeros_(final_layer.bias)

        return nn.Sequential(
            init_layer,
            nn.Tanh(),
            final_layer,
        )

    def forward(self, x):
        logits = self.neural_network(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (neural_network): Sequential(
    (0): Linear(in_features=784, out_features=32, bias=True)
    (1): Tanh()
    (2): Linear(in_features=32, out_features=10, bias=True)
  )
)


In [8]:
logits = model(datas[0].flattened_pixels)
predicted_probabilities = nn.Softmax(dim=0)(logits)
predicted_label = predicted_probabilities.argmax()

print(f'{predicted_label=}, {datas[0].label}')

predicted_label=tensor(6), 5


In [9]:
from datetime import datetime

In [10]:
losses: list[float] = []
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [11]:
for i, data in enumerate(datas):
  logits = model(data.flattened_pixels)
  predicted_probabilities = nn.Softmax(dim=0)(logits)
  predicted_label = predicted_probabilities.argmax()
  loss = torch.nn.functional.cross_entropy(logits, data.label_to_one_hot())
  losses.append(loss.item())
  
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if i % 100 == 0 and i > 0:
    window = losses[-100:]
    print(f'{datetime.now().strftime("%H:%M:%S")} {i=} mean_t100_loss={sum(window)/len(window):.4f}')

15:59:40 i=100 mean_t100_loss=2.1337
15:59:40 i=200 mean_t100_loss=1.8038
15:59:40 i=300 mean_t100_loss=1.5367
15:59:40 i=400 mean_t100_loss=1.2514
15:59:40 i=500 mean_t100_loss=1.1465
15:59:40 i=600 mean_t100_loss=1.1340
15:59:40 i=700 mean_t100_loss=1.0417
15:59:40 i=800 mean_t100_loss=0.8443
15:59:40 i=900 mean_t100_loss=0.8432
15:59:40 i=1000 mean_t100_loss=0.7530
15:59:40 i=1100 mean_t100_loss=0.9098
15:59:40 i=1200 mean_t100_loss=0.7389
15:59:41 i=1300 mean_t100_loss=0.7296
15:59:41 i=1400 mean_t100_loss=0.6630
15:59:41 i=1500 mean_t100_loss=0.5480
15:59:41 i=1600 mean_t100_loss=0.6141
15:59:41 i=1700 mean_t100_loss=0.4736
15:59:41 i=1800 mean_t100_loss=0.4100
15:59:41 i=1900 mean_t100_loss=0.4713
15:59:41 i=2000 mean_t100_loss=0.4620
15:59:41 i=2100 mean_t100_loss=0.5420
15:59:41 i=2200 mean_t100_loss=0.3704
15:59:41 i=2300 mean_t100_loss=0.3819
15:59:41 i=2400 mean_t100_loss=0.4139
15:59:41 i=2500 mean_t100_loss=0.4613
15:59:41 i=2600 mean_t100_loss=0.2778
15:59:41 i=2700 mean_

In [12]:
logits = model(datas[0].flattened_pixels)
predicted_probabilities = nn.Softmax(dim=0)(logits)
predicted_label = predicted_probabilities.argmax()

print(f'{predicted_label=}, {datas[0].label}')

predicted_label=tensor(5), 5


In [13]:
for name, parameter in model.named_parameters():
  print(f"Layer: {name} | Size: {parameter.size()} | Values : {parameter[:2]} \n")

Layer: neural_network.0.weight | Size: torch.Size([32, 784]) | Values : tensor([[ 0.0382, -0.0217,  0.0378,  ..., -0.0342, -0.0021, -0.0756],
        [ 0.0021,  0.0080, -0.0528,  ..., -0.0053, -0.0343, -0.0413]],
       grad_fn=<SliceBackward0>) 

Layer: neural_network.0.bias | Size: torch.Size([32]) | Values : tensor([-0.0551, -0.3110], grad_fn=<SliceBackward0>) 

Layer: neural_network.2.weight | Size: torch.Size([10, 32]) | Values : tensor([[-0.1251,  0.8419, -0.9050, -0.5148,  0.3288, -0.0242,  0.2990, -0.1886,
          0.4004, -0.5485, -0.3991, -0.7713,  0.1760, -1.3730,  0.8004,  0.0016,
          0.1803, -0.2286, -0.7039,  0.6553,  0.0389, -0.3569,  0.3367, -0.4625,
          0.0100,  0.0671, -0.6227,  0.4826,  0.5408, -0.4429,  0.4868,  0.0550],
        [ 0.4978, -0.6501,  0.5702,  0.0420,  0.5377,  0.0480,  0.3907,  0.4451,
          0.7893,  0.8551, -0.4590,  0.1987, -0.6490,  0.0350, -0.5749,  0.8024,
          0.5812,  0.3662,  0.6433, -0.2508,  0.4333,  0.5902,  0.5769,  0

In [14]:
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'pyparsing'

## Test

In [15]:
lines: list[str] = []
datas_test: list[Data] = []

with open('data/MNIST_CSV/mnist_test.csv') as f:
  lines = f.readlines()

for line in lines:
  datas_test.append(Data.from_csv_row([int(num) for num in line.split(',')]))

In [16]:
# TODO: figure out how
model.neural_network.detach()

AttributeError: 'Sequential' object has no attribute 'detach'

In [17]:
datas_test[0].label

7

In [18]:
logits = model(datas_test[0].flattened_pixels)
predicted_probabilities = nn.Softmax(dim=0)(logits)
for label, probability in enumerate(predicted_probabilities):
  print(f'P({label}) = {probability.item()}')
predicted_label = predicted_probabilities.argmax()

print(f'{predicted_label=}, {datas_test[0].label}')

P(0) = 2.536648571549449e-05
P(1) = 9.844629289545992e-08
P(2) = 5.7655252021504566e-05
P(3) = 4.349515802459791e-05
P(4) = 2.3367920221062377e-07
P(5) = 1.7140129102699575e-06
P(6) = 1.756893119875258e-08
P(7) = 0.9996834993362427
P(8) = 2.697794116102159e-06
P(9) = 0.00018524912593420595
predicted_label=tensor(7), 7


In [26]:
class TestOutcome:
  def __init__(
      self,
      test_case_index: int,
      correct_label: int,
      predicted_label: int,
      predicted_label_to_probability: dict[int, float],
  ):
    test_case_index = test_case_index
    self.correct_label = correct_label
    self.predicted_label = predicted_label
    self.predicted_label_to_probability = predicted_label_to_probability

  def __repr__(self) -> str:
    return f'{self.__class__.__name__}({self.__dict__})'

In [27]:
test_outcomes: list[TestOutcome] = []
incorrect_test_outcomes: list[TestOutcome] = []

In [28]:
from datetime import datetime

In [29]:
for i in range(len(datas_test)):
  # compute prediction
  data = datas_test[i]
  logits = model(data.flattened_pixels)
  predicted_probabilities = nn.Softmax(dim=0)(logits)

  predicted_label_to_probability: dict[int, float] = {}
  for label, probability in enumerate(predicted_probabilities):
    predicted_label_to_probability[label] = probability.item()
  predicted_label = predicted_probabilities.argmax()

  predicted_label = max(predicted_label_to_probability.keys(), key=lambda label: predicted_label_to_probability[label])
  
  test_outcome = TestOutcome(
      test_case_index=i,
      correct_label=data.label,
      predicted_label=predicted_label,
      predicted_label_to_probability=predicted_label_to_probability,
  )
  
  # update state
  test_outcomes.append(test_outcome)
  if test_outcome.correct_label != test_outcome.predicted_label:
    incorrect_test_outcomes.append(test_outcome)

  if i % 100 == 0 and i > 0:
    print(f'{datetime.now().strftime("%H:%M:%S")} {i=} current_score={(i - len(incorrect_test_outcomes)) / i:.4f}')

16:01:46 i=100 current_score=0.9700
16:01:46 i=200 current_score=0.9700
16:01:46 i=300 current_score=0.9667
16:01:46 i=400 current_score=0.9450
16:01:46 i=500 current_score=0.9440
16:01:46 i=600 current_score=0.9450
16:01:46 i=700 current_score=0.9429
16:01:46 i=800 current_score=0.9437
16:01:46 i=900 current_score=0.9422
16:01:46 i=1000 current_score=0.9390
16:01:46 i=1100 current_score=0.9373
16:01:46 i=1200 current_score=0.9333
16:01:46 i=1300 current_score=0.9285
16:01:46 i=1400 current_score=0.9293
16:01:46 i=1500 current_score=0.9247
16:01:46 i=1600 current_score=0.9231
16:01:46 i=1700 current_score=0.9206
16:01:46 i=1800 current_score=0.9189
16:01:46 i=1900 current_score=0.9184
16:01:46 i=2000 current_score=0.9180
16:01:46 i=2100 current_score=0.9171
16:01:46 i=2200 current_score=0.9150
16:01:46 i=2300 current_score=0.9161
16:01:46 i=2400 current_score=0.9175
16:01:46 i=2500 current_score=0.9180
16:01:46 i=2600 current_score=0.9196
16:01:46 i=2700 current_score=0.9215
16:01:46 i

In [30]:
test_outcome.correct_label, test_outcome.predicted_label

(6, 6)